# V distillation pilot on CREMA-D: an HSE-only teacher mixed into the video-only target (Kaggle, 2× T4)

**Question.** Can the specialist's better reading of the video-only votes be moved into Qwen itself, so that HSEmotion
is not needed at inference, while the audio-visual reading is preserved? (Design locked 11/11/2026.)

**Arms** (`ARM`), identical except for the target of the V reading:
- `cont`: continued training on the human votes, per clip `1/2 CE(p_AV, q_AV) + 1/2 CE(p_V, q_V)`.
- `distill`: the same with the V target replaced by `p~_V = 2/3 p_V + 1/3 q_T` (alpha = 1/3, temperature 1), where
  `q_T` is the frozen HSE-only teacher's distribution for that clip, cross-fitted within this fold's training actors
  (`analysis/distill_teacher.py`): the teacher that predicts a clip never saw its actor. The AV target is never the
  teacher. Both arms keep the AV and V task weights equal (1/2 each); only the V target differs.

**Locked settings** (as the transfer pilot): start from the `s1_noI` adapter of the same outer fold; one extra epoch,
last checkpoint, seed 0, the clip order of Stage 1 seed 0; fresh AdamW and scheduler (LoRA LR 1e-4, linear warm-up over
3% of the steps, then linear decay; weight decay 0; gradient clipping at norm 1); 8 clips per optimizer step. No
HSEmotion input at inference. Scored: the held-out actors under AV, V and A.

**Checks printed during the run.** Before training: the start adapter reproduces the `s1_noI` scores of 6 held-out
clips under AV and V; every training clip has a teacher distribution; the label order of the teacher file matches;
the mixed targets sum to 1.

**Reading** (fixed before the runs; `analysis/distill_compare.py`): `distill` minus `cont`, paired actor bootstrap,
three folds pooled. Transfer: JSD V (main), partial V (needed to claim a better disagreement ranking). AV preservation
(margins set by the project): upper 95% bound of delta JSD AV < +0.002 and lower bound of delta partial AV > -0.02.

**Running it** (one notebook run per arm and fold)
1. *Accelerator*: **GPU T4 x2**; *Internet*: **on**; *Add input*: `cremad_mm/`, the output of the `s1_noI` Stage 1 run of
   the same fold (`stage1_s1_noI_f<FOLD>_s0/`), and the teacher files (`distill_teacher/teacher_v_f<FOLD>.csv`).
2. Set `ARM` and `FOLD`. First `LIMIT = 10` for each arm (technical check), then the full run.
3. **Save & Run All (Commit)**; if it stops at `TIME_BUDGET_H`, attach this notebook's own output and commit again.
4. Results: `/kaggle/working/distill_<ARM>_f<FOLD>_s0/` with `scores.jsonl`, `state.json`, `start_check.csv`.

In [ ]:
# peft added for LoRA; the rest as in the other notebooks.
%pip install -q "transformers==5.20.1" "qwen-omni-utils==0.0.9" "peft==0.21.2" audioread librosa
# Kaggle ships torchao 0.10; peft 0.21 raises an ImportError for torchao < 0.16 even when it is not used.
%pip uninstall -y -q torchao

In [ ]:
import glob
import json
import os
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-Omni-7B")  # env override: local tests with a tiny model
LOAD_IN_4BIT = False         # True: 4-bit NF4 weights (needs bitsandbytes), if fp16 does not fit
MAX_MEMORY = {0: "12GiB", 1: "14GiB"}  # leave headroom on GPU 0 for activations
FRAME_MAX_PIXELS = 448 * 252           # per-frame pixel cap passed to the Qwen video processor
DRY_RUN = os.environ.get("DRY_RUN") == "1"  # build inputs only, no model (for local checks)
SEED = 0

# Folder made by analysis/preprocess_chsims.py; found automatically under /kaggle/input.
DATA_DIR = os.environ.get("CHSIMS_MM_DIR")
if DATA_DIR is None:
    hits = [h for k in range(6) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + "manifest.csv"))]  # no ** : media folders make it slow
    assert hits, "No manifest.csv under /kaggle/input: attach the preprocessed dataset."
    DATA_DIR = os.path.dirname(hits[0])
DATA_DIR = Path(DATA_DIR)

# Folder made by kaggle/prep_cremad.ipynb (analysis/preprocess_cremad.py); found automatically under /kaggle/input.
DATA_DIR = os.environ.get("CREMAD_MM_DIR")
if DATA_DIR is None:
    hits = [h for k in range(6) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + "cremad_mm/manifest.csv"))]
    assert hits, "No cremad_mm/manifest.csv under /kaggle/input: attach the output of prep_cremad.ipynb."
    DATA_DIR = os.path.dirname(hits[0])
DATA_DIR = Path(DATA_DIR)

ARM = os.environ.get("ARM", "cont")          # cont (human votes) | distill (V target mixed with the teacher)
PHASE = os.environ.get("PHASE", "outer")     # outer: score the outer fold | inner: lambda search on held-out training actors
FOLD = int(os.environ.get("FOLD", 0))        # outer fold, 0..N_OUTER-1
SEED = int(os.environ.get("SEED", 0))
VIEW = os.environ.get("VIEW", "A0")          # A0: 2 fps (Stage 1) | A1: 4 fps (needs the output of prep_cremad_views.ipynb)
S3 = ARM.startswith("s3")
LAMBDA_AV, LAMBDA_V, LAMBDA_A = 0.5, 0.5, 0.0  # per clip: 1/2 CE_AV + 1/2 CE_V (both arms); A not in the loss
LAMBDA_I = 0.0
# Stage 3: ranking loss on an ambiguity head (AV reading), weight chosen on the inner split of fold 0 from {0.1, 0.5}
LAMBDA_AMB = float(os.environ.get("LAMBDA_AMB", 0.1)) if ARM in ("s3", "s3_frozenvis", "s3_noV") else 0.0
TRAIN_VISUAL = ARM in ("s3", "s3_norank", "s3_noV")  # train the merger + LoRA on the last VIS_BLOCKS vision blocks
VIS_BLOCKS, MERGER_LR = 8, 1e-5
RANK_DELTA, RANK_QUEUE, MIN_SENTENCE_PAIRS = 0.15, 256, 8  # pairs: same emotion, |A_i - A_j| >= RANK_DELTA; same
                                                          # sentence too when at least MIN_SENTENCE_PAIRS of them exist
N_OUTER, INNER_VAL_SHARE, SPLIT_SEED = 3, 0.15, 0
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = r"^model\.layers\.\d+\.self_attn\.(q|k|v|o)_proj$"
LR, HEAD_LR = 1e-4, 1e-3
EPOCHS = 1
GRAD_ACCUM = 8
WARMUP = 0.03
CKPT_EVERY = 200
TRAIN_MAX_MEMORY = {0: "10GiB", 1: "14GiB"}
TIME_BUDGET_H = 11.0
LIMIT = None                 # e.g. 10 for a quick check: 10 training clips and 10 scored clips
assert ARM in ("cont", "distill")
assert PHASE == "outer" and 0 <= FOLD < N_OUTER and VIEW == "A0"
TRAIN_CONDS = ("AV", "V")
STAGE3 = S3 or VIEW != "A0"
SCORE_CONDS = ("V",) if ARM == "v_only" else ("AV", "V") if STAGE3 else ("AV", "V", "A")
if STAGE3:
    RUN_NAME = f"stage3_{ARM}_{VIEW}_f{FOLD}_s{SEED}" + (f"_inner_amb{LAMBDA_AMB:g}" if PHASE == "inner" else "")
else:
    RUN_NAME = f"distill_{ARM}_f{FOLD}_s{SEED}"
TAG = f"{MODEL_ID}+{RUN_NAME}"
OUT_DIR = Path(os.environ.get("OUT_DIR", f"/kaggle/working/{RUN_NAME}"))
T_START = time.time()
VIEWS_DIR = os.environ.get("CREMAD_VIEWS_DIR")
if VIEW == "A1" and VIEWS_DIR is None:
    hits = [h for k in range(9) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + "cremad_views/views.csv"))]
    if not hits:  # attached as an archive that was not unpacked: unpack it outside the output folder
        import zipfile
        for z in (h for k in range(9) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + "*.zip"))):
            with zipfile.ZipFile(z) as zf:
                names = [n for n in zf.namelist() if n.endswith("cremad_views/views.csv") or n == "views.csv"]
                if names:
                    print("unpacking", z, "(about a minute)")
                    zf.extractall("/tmp/cremad_views_unzip")
                    hits = [os.path.join("/tmp/cremad_views_unzip", names[0])]
                    break
    assert hits, "No cremad_views/views.csv under /kaggle/input: attach the output of prep_cremad_views.ipynb."
    VIEWS_DIR = os.path.dirname(hits[0])

# ---- V distillation pilot (locked) ----
ALPHA = 1 / 3                    # teacher share of the V target in the distill arm (temperature 1)
START_ARM, N_START_CHECK = "s1_noI", 6
START_ADAPTER = os.environ.get("START_ADAPTER")
if START_ADAPTER is None:
    name = f"stage1_{START_ARM}_f{FOLD}_s0"
    hits = [h for k in range(6) for h in sorted(glob.glob("/kaggle/input/" + "*/" * k + f"{name}/last/adapter.pt"))]
    assert hits, f"No {name}/last/adapter.pt under /kaggle/input: attach the output of that Stage 1 run."
    START_ADAPTER = hits[0]
START_ADAPTER = Path(START_ADAPTER)
st = json.loads((START_ADAPTER.parent.parent / "state.json").read_text())
assert (st["arm"], st["phase"], st["fold"], st["seed"], st["train_done"]) == (START_ARM, "outer", FOLD, 0, True), st
TEACHER = os.environ.get("TEACHER")
if TEACHER is None:
    # anywhere under /kaggle/input (in a distill_teacher/ folder or at the root of the dataset)
    hits = sorted({h for k in range(8) for h in glob.glob("/kaggle/input/" + "*/" * k + f"teacher_v_f{FOLD}.csv")})
    if not hits:
        print("\n".join(sorted(glob.glob("/kaggle/input/*/*/*") + glob.glob("/kaggle/input/*/*/*/*"))[:80]))
    assert hits or ARM == "cont", f"No teacher_v_f{FOLD}.csv under /kaggle/input (folders listed above): attach the teacher dataset."
    assert len(hits) <= 1, f"several teacher files for fold {FOLD}: {hits}"
    TEACHER = hits[0] if hits else None
print("start adapter:", START_ADAPTER, "| teacher:", TEACHER)
assert SCORE_CONDS == ("AV", "V", "A") and not STAGE3
print("run:", RUN_NAME, "| view:", VIEW, "| lambdas V/A/I/amb:", LAMBDA_V, LAMBDA_A, LAMBDA_I, LAMBDA_AMB,
      "| train visual:", TRAIN_VISUAL, "| readings:", TRAIN_CONDS)

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| out:", OUT_DIR, "| dry run:", DRY_RUN)
print("free disk on / (GB):", round(shutil.disk_usage("/").free / 1e9, 1))
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i}", torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), "GiB")

## Load the model, add LoRA and the intensity heads

In [ ]:
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from qwen_omni_utils import process_mm_info
from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration

ON_GPU = torch.cuda.is_available()
processor = Qwen2_5OmniProcessor.from_pretrained(MODEL_ID)
kwargs = dict(dtype=torch.float16 if ON_GPU else torch.float32, attn_implementation="sdpa")
if ON_GPU:
    kwargs.update(device_map="auto", max_memory=TRAIN_MAX_MEMORY)
t0 = time.time()
base = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(MODEL_ID, **kwargs)
print(f"loaded in {time.time() - t0:.0f} s")
base.config.use_cache = False
base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
HIDDEN = {}


def keep_last(module, args):
    HIDDEN["h"] = args[0][:, -1]
    return (args[0][:, -1:],) + args[1:]


base.lm_head.register_forward_pre_hook(keep_last)
torch.manual_seed(SEED)
targets = LORA_TARGETS
if TRAIN_VISUAL:  # + LoRA on the attention (q, k, v, proj) of the last VIS_BLOCKS vision blocks
    n_vis = len(base.visual.blocks)
    ids = "|".join(str(i) for i in range(max(0, n_vis - VIS_BLOCKS), n_vis))
    targets = rf"^(model\.layers\.\d+\.self_attn\.(q|k|v|o)_proj|visual\.blocks\.({ids})\.attn\.(q|k|v|proj))$"
model = get_peft_model(base, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                        target_modules=targets, bias="none"))
lora_params = [p for p in model.parameters() if p.requires_grad]
HEAD_DEVICE = base.lm_head.weight.device
merger_params = []
if TRAIN_VISUAL:  # the merger (vision tokens -> language model) trained in full, in float32 like the LoRA weights
    merger = base.visual.merger
    merger.float()
    merger.register_forward_pre_hook(lambda module, args: (args[0].float(),) + args[1:])
    merger.register_forward_hook(lambda module, args, out: out.to(base.lm_head.weight.dtype))
    for p in merger.parameters():
        p.requires_grad_(True)
    merger_params = list(merger.parameters())


class IntensityHeads(torch.nn.Module):
    # one linear head per reading on the last hidden state: the raters' mean perceived intensity, standardised
    def __init__(self, d):
        super().__init__()
        self.norm = torch.nn.LayerNorm(d, elementwise_affine=False)
        self.out = torch.nn.ModuleDict({c: torch.nn.Linear(d, 1) for c in ("AV", "V", "A")})

    def forward(self, h, c):
        return self.out[c](self.norm(h)).squeeze(-1)


hidden_size = base.config.text_config.hidden_size if hasattr(base.config, "text_config") else base.config.hidden_size
heads = IntensityHeads(hidden_size).to(HEAD_DEVICE, torch.float32)


class AmbiguityHead(torch.nn.Module):
    # a ranking score s for the AV raters' disagreement, from the AV reading's last hidden state
    def __init__(self, d):
        super().__init__()
        self.norm = torch.nn.LayerNorm(d, elementwise_affine=False)
        self.out = torch.nn.Linear(d, 1)

    def forward(self, h):
        return self.out(self.norm(h)).squeeze(-1)


with torch.random.fork_rng(devices=[]):  # leaves the random state of the Stage 1 runs unchanged
    amb_head = AmbiguityHead(hidden_size).to(HEAD_DEVICE, torch.float32)
n_vis_lora = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and ".visual." in n)
print(f"LoRA trainable params: {sum(p.numel() for p in lora_params) / 1e6:.1f} M (vision: {n_vis_lora / 1e6:.2f} M) | "
      f"merger: {sum(p.numel() for p in merger_params) / 1e6:.1f} M | intensity heads: "
      f"{sum(p.numel() for p in heads.parameters()) / 1e3:.0f} k | readings trained: {TRAIN_CONDS}")
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i} allocated after load: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB")

## Start adapter

In [ ]:
res = set_peft_model_state_dict(model, torch.load(START_ADAPTER, map_location="cpu"))
assert not getattr(res, "unexpected_keys", []), res.unexpected_keys
print("loaded start adapter", START_ADAPTER)

## Prompt (as in the zero-shot CREMA-D run)

In [ ]:
LABELS = ["Anger", "Disgust", "Fear", "Happy", "Neutral", "Sad"]  # CREMA-D's six labels, as the raters saw them
DESCRIBE = {  # the three conditions the raters saw; no transcript (raters did not read the sentence)
    "AV": "the video frames and the audio track",
    "A": "only the audio track",
    "V": "only the video frames, without sound",
}
CONDITIONS = list(DESCRIBE)
ASK = f"Which emotion does the speaker express? Choose exactly one of: {', '.join(LABELS)}.\nAnswer with the label only."


def intro(condition):
    return f"You are given {DESCRIBE[condition]} of an actor saying a short sentence.\n"

SYSTEM = ("You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of "
          "perceiving auditory and visual inputs, as well as generating text and speech.")
label_first_ids = [processor.tokenizer.encode(l, add_special_tokens=False)[0] for l in LABELS]
assert len(set(label_first_ids)) == len(LABELS), "two labels share a first token"


def build_messages(row, condition, ask="label"):  # `ask` kept for the shared cells
    content = []
    if "V" in condition:
        frames = sorted(glob.glob(str(DATA_DIR / row["frames_dir"] / "*.jpg")))
        content.append({"type": "video", "video": [f"file://{f}" for f in frames],
                        "sample_fps": float(row["sample_fps"]), "max_pixels": FRAME_MAX_PIXELS})
    if "A" in condition:
        content.append({"type": "audio", "audio": str(DATA_DIR / row["audio_path"])})
    content.append({"type": "text", "text": intro(condition) + ASK})
    return [{"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
            {"role": "user", "content": content}]


def prepare(messages):
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    audios, images, videos, video_kwargs = process_mm_info(messages, use_audio_in_video=False, return_video_kwargs=True)
    fps = video_kwargs.pop("fps", [])  # qwen-omni-utils returns a per-video list; the processor wants one number
    if fps:
        video_kwargs["fps"] = float(fps[0])
    inputs = processor(text=text, audio=audios, images=images, videos=videos, return_tensors="pt",
                       padding=True, use_audio_in_video=False, **video_kwargs)
    if model is not None:
        inputs = {k: (v.to(model.device, dtype=model.dtype) if torch.is_floating_point(v) else v.to(model.device))
                  if torch.is_tensor(v) else v for k, v in inputs.items()}
    return inputs

## Clips, folds and targets

`p_AV`, `p_V`, `p_A`: each condition's vote distribution over the six labels (`s1_avlab`: `p_V` = `p_A` = `p_AV`).
`I_AV`, `I_V`, `I_A`: the mean intensity rating (1-100) of that condition's raters, from CREMA-D's
`finishedResponses.csv`, standardised with the training clips' mean and SD per condition (`zI_*`), so that the squared
error starts near 1, on the scale of the cross-entropies. Folds are made from actors with a fixed seed, independent of
`SEED`.

In [ ]:
import io
import urllib.request

manifest = pd.read_csv(DATA_DIR / "manifest.csv", dtype={"video_id": str})
clips = manifest[manifest["status"] == "ok"].reset_index(drop=True)
url = "https://raw.githubusercontent.com/CheyneyComputerScience/CREMA-D/master/finishedResponses.csv"
resp = pd.read_csv(io.BytesIO(urllib.request.urlopen(url, timeout=120).read()), low_memory=False)
resp = resp.drop_duplicates(["localid", "clipName", "queryType"])
resp["level"] = pd.to_numeric(resp["respLevel"], errors="coerce")
inten = resp.groupby(["clipName", "queryType"])["level"].mean().unstack().rename(columns={1: "I_A", 2: "I_V", 3: "I_AV"})
clips = clips.merge(inten, left_on="uid", right_index=True, how="left")
for c, k in (("AV", "av"), ("V", "v"), ("A", "a")):
    votes = clips[[f"{k}_{l}" for l in LABELS]].to_numpy(float)
    clips[f"p_{c}"] = list(votes / votes.sum(1, keepdims=True))
pav = np.stack(clips["p_AV"])  # A^AV: normalised entropy of the AV votes (the ranking target)
clips["A_AV"] = -np.where(pav > 0, pav * np.log(np.where(pav > 0, pav, 1)), 0).sum(1) / np.log(len(LABELS))
if VIEW == "A1":  # 4 fps frames for every reading (build_messages joins DATA_DIR with an absolute path unchanged)
    views = pd.read_csv(Path(VIEWS_DIR) / "views.csv")
    views = views[views["status"].isin(["ok", "face_fallback"])]
    n0 = len(clips)
    clips = clips.merge(views[["uid", "frames4_dir", "sample_fps4"]], on="uid")
    clips["frames_dir"] = [str(Path(VIEWS_DIR) / d) for d in clips["frames4_dir"]]
    clips["sample_fps"] = clips["sample_fps4"]
    print(f"A1: 4 fps frames for {len(clips)} of {n0} clips")
if ARM == "s1_avlab":
    clips["p_V"], clips["p_A"] = clips["p_AV"], clips["p_AV"]

rng = np.random.default_rng(SPLIT_SEED)
actor_sex = clips.groupby("actor")["Sex"].first()
outer = {}
for _, group in actor_sex.groupby(actor_sex):
    for k, a in enumerate(rng.permutation(group.index.to_numpy())):
        outer[int(a)] = k % N_OUTER
clips["outer_fold"] = clips["actor"].map(outer)
train_actors = sorted(a for a, f in outer.items() if f != FOLD)
inner_val = set(np.random.default_rng(SPLIT_SEED + 1 + FOLD).permutation(train_actors)[:max(1, round(INNER_VAL_SHARE * len(train_actors)))])
if PHASE == "outer":
    train_df = clips[clips["outer_fold"] != FOLD].reset_index(drop=True)
    eval_df = clips[clips["outer_fold"] == FOLD].reset_index(drop=True)
else:
    trainval = clips[clips["outer_fold"] != FOLD]
    train_df = trainval[~trainval["actor"].isin(inner_val)].reset_index(drop=True)
    eval_df = trainval[trainval["actor"].isin(inner_val)].reset_index(drop=True)
if LIMIT:
    train_df, eval_df = train_df.head(LIMIT), eval_df.head(LIMIT)
assert not set(train_df["actor"]) & set(eval_df["actor"])
I_STATS = {c: (train_df[f"I_{c}"].mean(), train_df[f"I_{c}"].std()) for c in ("AV", "V", "A")}
for df in (train_df, eval_df):
    for c, (mu, sd) in I_STATS.items():
        df[f"zI_{c}"] = (df[f"I_{c}"] - mu) / sd
print(f"train {len(train_df)} clips / {train_df['actor'].nunique()} actors | scored {len(eval_df)} clips / "
      f"{eval_df['actor'].nunique()} actors | intensity missing: {int(train_df[['I_AV', 'I_V', 'I_A']].isna().any(axis=1).sum())}")

## V targets (teacher)

In [ ]:
# V target of each training clip: the human votes (cont) or 2/3 votes + 1/3 cross-fitted teacher (distill).
train_df["p_Vt"] = train_df["p_V"]
if TEACHER is not None:
    t = pd.read_csv(TEACHER)
    assert list(t.columns[2:]) == LABELS, (list(t.columns), LABELS)  # label order of the teacher file
    t = t[t.role == "train"].set_index("uid")
    missing = [u for u in train_df["uid"] if u not in t.index]
    assert not missing, f"{len(missing)} training clips without a teacher distribution, e.g. {missing[:3]}"
    assert not set(eval_df["uid"]) & set(t.index), "teacher train rows include held-out clips"
    QT = {u: t.loc[u, LABELS].to_numpy(float) for u in train_df["uid"]}
    if ARM == "distill":
        train_df["p_Vt"] = [(1 - ALPHA) * np.asarray(p) + ALPHA * QT[u] for p, u in zip(train_df["p_V"], train_df["uid"])]
    q = np.stack([QT[u] for u in train_df["uid"]])
    pv = np.stack(train_df["p_V"])
    print(f"teacher on the training clips: mean JSD to the V votes {float(np.mean([0.5 * (np.sum(a * np.log(np.clip(a, 1e-12, None) / np.clip((a + b) / 2, 1e-12, None))) + np.sum(b * np.log(np.clip(b, 1e-12, None) / np.clip((a + b) / 2, 1e-12, None)))) for a, b in zip(pv, q)])):.4f}")
tgt = np.stack(train_df["p_Vt"])
assert np.allclose(tgt.sum(1), 1, atol=1e-6) and (tgt >= 0).all()
print(f"V targets: {ARM} | mean |p~_V - p_V| {float(np.abs(tgt - np.stack(train_df['p_V'])).mean()):.4f}")
TARGET_COL = {"AV": "p_AV", "V": "p_Vt", "A": "p_A"}

## One clip: readings, losses, back-propagation

Each reading is back-propagated right after its forward pass (the readings share no computation, so the gradient is
the same as for the summed loss).

In [ ]:
WEIGHT = {"AV": LAMBDA_AV, "V": LAMBDA_V, "A": LAMBDA_A}
QUEUE = []  # (detached AV hidden state, A^AV, emotion, sentence) of the last RANK_QUEUE training clips
RANK_BINS = [0.15, 0.25, 0.4]


def rank_loss(h, row):
    # pairwise logistic ranking of the current clip against queued clips, scored with the current head
    a, emo, sent = float(row["A_AV"]), row["intended"], row["sentence_code"]
    valid = [q for q in QUEUE if q[2] == emo and abs(q[1] - a) >= RANK_DELTA]
    same = [q for q in valid if q[3] == sent]
    pairs = same if len(same) >= MIN_SENTENCE_PAIRS else valid
    log = state["rank"]
    log["clips"] += 1
    log["pairs"] += len(pairs)
    log["no_pair"] += not pairs
    log["sentence_level"] += bool(pairs) and pairs is same
    for q in pairs:
        log["delta_bins"][int(np.searchsorted(RANK_BINS, abs(q[1] - a), side="right")) - 1] += 1
    QUEUE.append((h.detach(), a, emo, sent))
    del QUEUE[:-RANK_QUEUE]
    if not pairs:
        return torch.zeros((), device=h.device)
    s_i = amb_head(h)
    s_j = amb_head(torch.stack([q[0] for q in pairs]))
    r = torch.tensor([np.sign(a - q[1]) for q in pairs], dtype=torch.float32, device=h.device)
    return torch.nn.functional.softplus(-r * (s_i - s_j)).mean()


def reading(row, condition):
    inputs = prepare(build_messages(row, condition))
    logits = model(**inputs, use_cache=False).logits[0, -1].float()
    return logits, HIDDEN["h"][0].to(HEAD_DEVICE, torch.float32)


def train_clip(row, scale):
    out = {}
    for c in TRAIN_CONDS:
        logits, h = reading(row, c)
        z = logits[label_first_ids]
        target = torch.tensor(np.asarray(row[TARGET_COL[c]], dtype=float), dtype=torch.float32, device=z.device)
        ce = -(target * torch.log_softmax(z, dim=-1)).sum()
        loss = WEIGHT[c] * ce.to(HEAD_DEVICE)
        if LAMBDA_I > 0 and not np.isnan(row[f"zI_{c}"]):
            loss = loss + LAMBDA_I * (heads(h, c) - float(row[f"zI_{c}"])) ** 2
        if c == "AV" and LAMBDA_AMB > 0:
            loss = loss + LAMBDA_AMB * rank_loss(h, row)
        scaler.scale(loss * scale).backward()
        out[c] = ce.item()
    return out

## Check before training

In [ ]:
# Before training: the start adapter reproduces the s1_noI scores of the same held-out clips.
ref = pd.DataFrame([json.loads(l) for l in open(START_ADAPTER.parent.parent / "scores.jsonl", encoding="utf-8") if l.strip()])
ref = ref[ref.get("error", pd.Series(index=ref.index, dtype=object)).isna()]
ref = ref.drop_duplicates(["uid", "condition"], keep="last").set_index(["uid", "condition"])
lcols = [f"logit_{l}" for l in LABELS]
rows = []
model.eval()
for _, row in eval_df[eval_df["uid"].isin(ref.index.get_level_values(0))].head(N_START_CHECK).iterrows():
    for c in ("AV", "V"):
        with torch.inference_mode():
            out = reading(row, c)[0][label_first_ids].cpu().numpy()
        r = ref.loc[(row["uid"], c), lcols].to_numpy(float)
        rows.append({"uid": row["uid"], "condition": c, "max |logit diff| vs s1_noI": float(np.abs(out - r).max()),
                     "same top label": bool(out.argmax() == r.argmax())})
model.train()
check = pd.DataFrame(rows)
check.to_csv(OUT_DIR / "start_check.csv", index=False)
print(check.to_string())
assert len(check) and check["max |logit diff| vs s1_noI"].median() < 0.05 and check["same top label"].all(), "start adapter does not reproduce s1_noI"

## Checkpoints and resume (as in RQ3/RQ4)

In [ ]:
groups = [{"params": lora_params, "lr": LR}, {"params": list(heads.parameters()) + list(amb_head.parameters()), "lr": HEAD_LR}]
if merger_params:
    groups.append({"params": merger_params, "lr": MERGER_LR})
optimizer = torch.optim.AdamW(groups, weight_decay=0.0)
steps_per_epoch = -(-len(train_df) // GRAD_ACCUM)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = max(1, int(WARMUP * total_steps))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda s: min((s + 1) / warmup_steps, max(0.0, (total_steps - s) / max(1, total_steps - warmup_steps))))
scaler = torch.amp.GradScaler("cuda", enabled=ON_GPU)
IDENT = {"arm": ARM, "phase": PHASE, "fold": FOLD, "seed": SEED, "lambdas": [LAMBDA_V, LAMBDA_A, LAMBDA_I],
         "sizes": [len(train_df), len(eval_df)]}
IDENT.update(start=START_ADAPTER.parent.parent.name, alpha=ALPHA if ARM == "distill" else 0.0,
             loss_weights=[LAMBDA_AV, LAMBDA_V, LAMBDA_A])
if STAGE3:
    IDENT.update(view=VIEW, lambda_amb=LAMBDA_AMB, train_visual=TRAIN_VISUAL)
state = {**IDENT, "epoch": 0, "pos": 0, "opt_step": 0, "loss_sum": {}, "loss_n": 0, "oom": 0, "history": [],
         "train_done": False, "rank": {"clips": 0, "pairs": 0, "no_pair": 0, "sentence_level": 0, "delta_bins": [0, 0, 0]}}


def read_state(folder):
    path = Path(folder) / "state.json"
    return json.loads(path.read_text()) if path.exists() else None


def save_weights(folder):
    torch.save(get_peft_model_state_dict(model), folder / "adapter.pt")
    torch.save(heads.state_dict(), folder / "heads.pt")
    if STAGE3:
        torch.save(amb_head.state_dict(), folder / "amb_head.pt")
    if merger_params:
        torch.save(base.visual.merger.state_dict(), folder / "merger.pt")


def load_weights(folder):
    set_peft_model_state_dict(model, torch.load(folder / "adapter.pt", map_location="cpu"))
    heads.load_state_dict(torch.load(folder / "heads.pt", map_location="cpu"))
    if STAGE3:
        amb_head.load_state_dict(torch.load(folder / "amb_head.pt", map_location="cpu"))
    if merger_params:
        base.visual.merger.load_state_dict(torch.load(folder / "merger.pt", map_location="cpu"))


def save_checkpoint():
    tmp = OUT_DIR / "last.tmp"
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir()
    save_weights(tmp)
    torch.save({"optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict()}, tmp / "optim.pt")
    shutil.rmtree(OUT_DIR / "last", ignore_errors=True)
    tmp.rename(OUT_DIR / "last")
    state["saved_at"] = time.time()
    (OUT_DIR / "state.json").write_text(json.dumps(state, indent=1))


def out_of_time():
    return (time.time() - T_START) / 3600 > TIME_BUDGET_H


def same_run(s):
    return all(s.get(k) == v for k, v in IDENT.items())


if read_state(OUT_DIR) is None:
    found = [(s, Path(p).parent) for p in (h for k in range(6) for h in glob.glob("/kaggle/input/" + "*/" * k + f"{RUN_NAME}/state.json"))
             if (s := read_state(Path(p).parent))]
    usable = [(s, d) for s, d in found if same_run(s)]
    if len(usable) < len(found):
        print(f"ignoring {len(found) - len(usable)} earlier run(s) with other settings")
    if usable:
        src = max(usable, key=lambda x: x[0].get("saved_at", 0))[1]
        shutil.copytree(src, OUT_DIR, dirs_exist_ok=True)
        print("copied earlier run from", src)

saved = read_state(OUT_DIR)
if saved and same_run(saved):
    state.update(saved)
    if (OUT_DIR / "last" / "adapter.pt").exists():
        load_weights(OUT_DIR / "last")
        opt = torch.load(OUT_DIR / "last" / "optim.pt", map_location="cpu")
        optimizer.load_state_dict(opt["optimizer"])
        scheduler.load_state_dict(opt["scheduler"])
        scaler.load_state_dict(opt["scaler"])
    print(f"resuming: clip {state['pos']}/{len(train_df)}, step {state['opt_step']}", "| training done" if state["train_done"] else "")
else:
    print(f"fresh start: {total_steps} optimizer steps")

## Train (one epoch; no epoch selection, so the scored clips play no part in training choices)

In [ ]:
def train():
    model.train()
    t_run, n_run = time.time(), 0
    while state["epoch"] < EPOCHS:
        order = np.random.default_rng(1000 * SEED + state["epoch"]).permutation(len(train_df))
        while state["pos"] < len(order):
            if out_of_time():
                save_checkpoint()
                return False
            batch = order[state["pos"]:state["pos"] + GRAD_ACCUM]
            for i in batch:
                try:
                    for c, v in train_clip(train_df.iloc[i], 1.0 / len(batch)).items():
                        state["loss_sum"][c] = state["loss_sum"].get(c, 0.0) + v
                    state["loss_n"] += 1
                except torch.cuda.OutOfMemoryError:
                    optimizer.zero_grad(set_to_none=True)
                    torch.cuda.empty_cache()
                    state["oom"] += 1
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(lora_params + list(heads.parameters()) + list(amb_head.parameters()) + merger_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            state["pos"] += len(batch)
            state["opt_step"] += 1
            n_run += len(batch)
            if state["pos"] % CKPT_EVERY == 0 or state["pos"] == len(order):
                save_checkpoint()
            if n_run <= 2 * GRAD_ACCUM or n_run in (48, 200) or state["opt_step"] % 50 == 0:
                sec = (time.time() - t_run) / n_run
                left = (EPOCHS * len(order) - state["epoch"] * len(order) - state["pos"]) * sec / 3600
                peak = max([torch.cuda.max_memory_allocated(d) / 2**30 for d in range(torch.cuda.device_count())] or [0])
                n = max(1, state["loss_n"])
                rk = state["rank"]
                rank_info = (f" | rank pairs/clip {rk['pairs'] / max(1, rk['clips']):.1f}, no pair {rk['no_pair'] / max(1, rk['clips']):.0%}, "
                             f"sentence-level {rk['sentence_level'] / max(1, rk['clips']):.0%}, |dA| bins {rk['delta_bins']}") if LAMBDA_AMB > 0 else ""
                print(f"clip {state['pos']}/{len(order)} | CE " + " ".join(f"{c} {v / n:.3f}" for c, v in state["loss_sum"].items())
                      + f" | {sec:.2f} s/clip, about {left:.1f} h left | peak {peak:.1f} GiB | OOM {state['oom']}" + rank_info)
        n = max(1, state["loss_n"])
        state["history"].append({"epoch": state["epoch"], **{f"train_CE_{c}": v / n for c, v in state["loss_sum"].items()},
                                 "session_hours": round((time.time() - T_START) / 3600, 2)})
        state.update(epoch=state["epoch"] + 1, pos=0, loss_sum={}, loss_n=0)
        save_checkpoint()
    state["train_done"] = True
    save_checkpoint()
    return True


finished = state["train_done"] or train()
print("training finished" if finished else "stopped at the time budget: add this notebook's output as an input and commit again")

## Score the held-out actors under AV, V and A

Same record format as the zero-shot CREMA-D runs (`model` = `TAG`), plus the intensity heads' outputs.

In [ ]:
SCORES = OUT_DIR / "scores.jsonl"


def key(rec):
    return (rec["model"], rec["uid"], rec["condition"], rec["task"])


n_new = 0
if finished:
    model.eval()
    records = {}
    if SCORES.exists():
        for line in open(SCORES, encoding="utf-8"):
            if line.strip() and "error" not in (rec := json.loads(line)):
                records[key(rec)] = rec
    with open(SCORES, "w", encoding="utf-8") as f:
        for rec in records.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    todo = [(row, c) for _, row in eval_df.iterrows() for c in SCORE_CONDS if (TAG, row["uid"], c, "score") not in records]
    print(f"{len(eval_df) * len(SCORE_CONDS)} items, {len(eval_df) * len(SCORE_CONDS) - len(todo)} already done, {len(todo)} to run")
    for row, c in todo:
        if out_of_time():
            print("stopped at the time budget: add this notebook's output as an input and commit again")
            break
        rec = {"model": TAG, "uid": row["uid"], "condition": c, "task": "score", "split": PHASE, "fold": FOLD}
        try:
            with torch.inference_mode():
                logits, h = reading(row, c)
                mu, sd = I_STATS[c]
                rec.update(lse=float(torch.logsumexp(logits, dim=-1)), intensity_pred=float(heads(h, c)) * sd + mu,
                           **{f"logit_{l}": float(v) for l, v in zip(LABELS, logits[label_first_ids].tolist())})
                if c == "AV" and LAMBDA_AMB > 0:
                    rec["amb_score"] = float(amb_head(h))
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            rec["error"] = "oom"
        with open(SCORES, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        n_new += 1
    print(f"scored {n_new} items this run")

## Quick check (the analysis script does the real evaluation)

In [ ]:
print(state["history"])
if SCORES.exists():
    s = pd.DataFrame([json.loads(l) for l in open(SCORES, encoding="utf-8") if l.strip()])
    s = s[s.get("error", pd.Series(index=s.index, dtype=object)).isna()]
    z = s[[f"logit_{l}" for l in LABELS]].to_numpy()
    q = np.exp(z - z.max(1, keepdims=True))
    q /= q.sum(1, keepdims=True)
    s["U"] = -(q * np.log(np.clip(q, 1e-12, None))).sum(1) / np.log(len(LABELS))
    s = s.merge(eval_df[["uid", "p_AV", "p_V", "p_A", "A_AV"]], on="uid")
    if "amb_score" in s:
        part = s[s["condition"] == "AV"]
        print("AV", len(part), "clips | Spearman(amb_score, A^AV):",
              round(pd.Series(part["amb_score"].to_numpy()).corr(pd.Series(part["A_AV"].to_numpy()), method="spearman"), 3))
    for c in SCORE_CONDS:
        part = s[s["condition"] == c]
        p = np.stack(part[f"p_{c}"])
        h = -(p * np.log(np.clip(p, 1e-12, None))).sum(1) / np.log(len(LABELS))
        print(c, len(part), "clips | Spearman(U, human entropy, same condition):",
              round(pd.Series(part["U"].to_numpy()).corr(pd.Series(h), method="spearman"), 3))